# Preprocess raw JSONL logs into flat tables

Four sources → one set of CSVs tagged with a `source` column:

1. **`models`** — main system, cloud LLMs (`study/runs/models/results/`)
2. **`local_models`** — main system, local LLMs (`study/runs/local_models/results/`)
3. **`react_baseline`** — ReAct baseline, dense-only retrieval (`study/baselines/react-baseline/results/`)
4. **`norag`** — no-retrieval ablation *(placeholder; runs not produced yet)*

Outputs land in `study/preprocessed/`:
- `sessions.csv` — one row per JSONL log.
- `steps.csv` — one row per workflow step (per session).
- `tools.csv` — one row per tool call.
- `hashmatch_fields.csv` — per-field precision/recall/F1 (long format).
- `workflow_configs.csv` — retrieval + model config recorded at session start.
- `protocols/<session_id>.json` — actual protocol produced by that run.
- `expected/<case_id>.json` — ground-truth protocol per case.
- `skipped.csv` — logs the extractor couldn't process, with reasons.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd() if Path.cwd().name == "nli_engine" else Path.cwd().parents[0]
assert (REPO_ROOT / "pyproject.toml").exists(), f"expected repo root, got {REPO_ROOT}"
STUDY_DIR = REPO_ROOT / "study"
OUT_DIR = STUDY_DIR / "preprocessed"
MANIFEST_PATH = REPO_ROOT / "dataset" / "manifest.yaml"

sys.path.insert(0, str(STUDY_DIR))

from preprocess import SOURCES, get_source, process_source, write_tables

from nli_engine.eval.io.dataset import load_manifest

CASES = {c.id: c for c in load_manifest(MANIFEST_PATH)}
EXPECTED_CACHE: dict = {}

COMBINED: dict[str, list] = {
    k: []
    for k in ("sessions", "steps", "tools", "hashmatch_fields", "workflow_configs")
}
PROTOCOLS: dict = {}
SKIPPED: list = []


def _collect(name: str) -> None:
    src = get_source(name)
    out = process_source(src, CASES, expected_cache=EXPECTED_CACHE)
    for key in COMBINED:
        COMBINED[key].extend(out[key])
    PROTOCOLS.update(out["protocols"])
    SKIPPED.extend((src.name, fname, reason) for fname, reason in out["skipped"])
    print(
        f"[{name}] sessions={len(out['sessions'])} steps={len(out['steps'])} "
        f"tools={len(out['tools'])} hm_fields={len(out['hashmatch_fields'])} "
        f"skipped={len(out['skipped'])}"
    )
    if out["skipped"]:
        for fname, reason in out["skipped"][:5]:
            print(f"   skip: {fname}: {reason}")


print(f"repo root: {REPO_ROOT}")
print(f"cases loaded: {len(CASES)}")
print(f"sources: {[s.name for s in SOURCES]}")

## 1. `models` — main system, cloud LLMs

In [ ]:
_collect("models")

## 2. `local_models` — main system, local LLMs

Local runs often crash; failed logs show up with `has_protocol=False` in `sessions.csv`. That's a data point, not an error — don't filter them out at this stage.

In [ ]:
_collect("local_models")

## 3. `react_baseline` — ReAct baseline (dense-only retrieval)

Note: this workflow logs a single `baseline_agent` step per turn, so per-step breakdowns in `steps.csv` are not comparable across sources. Session-level totals (`total_input_tokens`, `total_latency_ms`) still are.

In [ ]:
_collect("react_baseline")

## 4. `norag` — no-retrieval ablation *(placeholder)*

Once you produce the ablation runs under `study/runs/norag/results/`, the cell below will pick them up. Until then it will report `skipped: results_dir missing` and continue.

In [ ]:
_collect("norag")

## Write tables

In [ ]:
write_tables(COMBINED, PROTOCOLS, EXPECTED_CACHE, SKIPPED, OUT_DIR)

print(f"wrote to {OUT_DIR}")
for name in ("sessions", "steps", "tools", "hashmatch_fields", "workflow_configs"):
    path = OUT_DIR / f"{name}.csv"
    n_lines = sum(1 for _ in path.open()) - 1 if path.stat().st_size else 0
    print(f"  {name}.csv: {n_lines} rows")
n_proto = sum(1 for p in (OUT_DIR / "protocols").iterdir() if p.suffix == ".json")
n_expected = sum(1 for p in (OUT_DIR / "expected").iterdir() if p.suffix == ".json")
print(f"  protocols/: {n_proto} files")
print(f"  expected/: {n_expected} files")

## Sanity check

In [ ]:
import pandas as pd

sessions = pd.read_csv(OUT_DIR / "sessions.csv")
print("rows per source × model:")
print(sessions.groupby(["source", "model"]).size().to_string())
print()
print("has_protocol rate per source:")
print(sessions.groupby("source")["has_protocol"].mean().to_string())
print()
print("hm_f1 mean per source (protocol-only):")
print(sessions[sessions.has_protocol].groupby("source")["hm_f1"].mean().to_string())